# 10d. Validation Recommender Evaluation

**Role of this notebook.** Compare recommendation policies on the same holdout candidate set and report ranking metrics against the realized true utility `u_star`.

**Steps.**

1. Load holdout candidates, baseline MLP scores, and debiased EM06 EU scores.
2. Construct two reference policies: oracle ranking by `u_star` and random ranking by a fixed random seed.
3. Compute per-user ranking metrics at multiple cutoffs.
4. Compute score correlations with true utility over all holdout candidate rows.
5. Produce side-by-side comparison tables for oracle, random, baseline, and debiased EM06.
6. Save summary metrics, per-user metrics, correlations, and a JSON report.

| Policy | Ranking score | Interpretation |
|---|---:|---|
| Oracle | `u_star` | upper bound that ranks candidates by realized true utility |
| Random | seeded uniform random score | lower benchmark that ignores all user/video/context information |
| Baseline MLP | `score_vanilla` | behavior-head MLP score from the vanilla recommender |
| Debiased EM06 EU MLP | `EU_hat` | predicted posterior expected utility from the structural debiased recommender |

All ranking metrics use `u_star` as the evaluation target. Model scores are used only to order candidates.


## Metrics

For user `i`, method `m`, and cutoff `k`, let `R_m(i,k)` be the top `k` videos selected from that user's holdout candidates. Let `u_ij` denote realized true utility `u_star`.

**Mean utility at k** measures welfare delivered by the recommendation list:

$$
\mathrm{Utility@k}(m)=\frac{1}{|I|}\sum_{i\in I}\frac{1}{k}\sum_{j\in R_m(i,k)}u_{ij}.
$$

**Regret at k** measures distance from the oracle ranking:

$$
\mathrm{Regret@k}(m)=\mathrm{Utility@k}(oracle)-\mathrm{Utility@k}(m).
$$

**Normalized lift over random** measures the share of the oracle-versus-random gain recovered by a method:

$$
\mathrm{Lift@k}(m)=
\frac{\mathrm{Utility@k}(m)-\mathrm{Utility@k}(random)}
{\mathrm{Utility@k}(oracle)-\mathrm{Utility@k}(random)}.
$$

**NDCG at k** rewards placing high-utility videos earlier. Because utility can be negative, the notebook first shifts utilities into nonnegative user-local gains:

$$
g_{ij}=u_{ij}-\min_{j'\in C_i}u_{ij'}.
$$

The discounted gain and normalized discounted gain are:

$$
\mathrm{DCG@k}(m,i)=\sum_{r=1}^{k}\frac{g_{i,j_r}}{\log_2(r+1)},
$$

$$
\mathrm{NDCG@k}(m)=\frac{1}{|I|}\sum_{i\in I}\frac{\mathrm{DCG@k}(m,i)}{\mathrm{IDCG@k}(i)}.
$$

**Top-decile precision, recall, and hit rate** define a relevant video as one in the user's top 10 percent by true utility:

$$
\mathrm{Precision@k}(m)=\frac{1}{|I|}\sum_{i\in I}\frac{|R_m(i,k)\cap T_i|}{k}.
$$

$$
\mathrm{Recall@k}(m)=\frac{1}{|I|}\sum_{i\in I}\frac{|R_m(i,k)\cap T_i|}{|T_i|}.
$$

$$
\mathrm{HitRate@k}(m)=\frac{1}{|I|}\sum_{i\in I}1\{|R_m(i,k)\cap T_i|>0\}.
$$

**Oracle overlap at k** checks exact agreement with the oracle top-k set:

$$
\mathrm{Overlap@k}(m)=\frac{1}{|I|}\sum_{i\in I}\frac{|R_m(i,k)\cap R_{oracle}(i,k)|}{k}.
$$


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
OUT_DIR = ROOT / 'python_code_new' / 'outputs' / 'validation_simulation'

HOLDOUT_PATH = OUT_DIR / 'validation_holdout_candidates.parquet'
BASELINE_PATH = OUT_DIR / 'validation_baseline_holdout_scores.parquet'
DEBIASED_EM06_PATH = OUT_DIR / 'validation_debiased_em06_holdout_scores.parquet'
BASELINE_METRICS_PATH = OUT_DIR / 'validation_baseline_metrics.json'
DEBIASED_EM06_METRICS_PATH = OUT_DIR / 'validation_debiased_em06_metrics.json'

SUMMARY_CSV = OUT_DIR / 'validation_recsys_ranking_metrics_summary.csv'
PER_USER_PARQUET = OUT_DIR / 'validation_recsys_ranking_metrics_per_user.parquet'
CORR_CSV = OUT_DIR / 'validation_recsys_score_correlations.csv'
EVAL_JSON = OUT_DIR / 'validation_recsys_evaluation_metrics.json'

KS = [1, 5, 10, 20, 50, 100]
RELEVANCE_TOP_FRAC = 0.10
RANDOM_SEED = 20260627

for path in [HOLDOUT_PATH, BASELINE_PATH, DEBIASED_EM06_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print('output dir:', OUT_DIR)


output dir: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation


## 1. Load Holdout Scores

This section merges all candidate-level scores on `user_id` and `video_id`.

| Column | Meaning |
|---|---|
| `u_star` | realized true utility used for evaluation |
| `score_vanilla` | baseline recommender score |
| `EU_hat` | debiased EM06 predicted expected utility |
| `random_score` | seeded random reference score |
| `oracle_score` | equal to `u_star`, used for oracle ranking |

The merged table keeps one row per user-video holdout candidate. Missing model scores are treated as an error because all methods must be evaluated on the same candidate set.


In [2]:
holdout_cols = ['user_id', 'video_id', 'u_star']
optional_diversity_cols = ['cat_idx_int', 'i_cat_level1_id', 'i_cat_level2_id', 'i_cat_level3_id']

holdout_schema_cols = pd.read_parquet(HOLDOUT_PATH).columns.tolist()
read_cols = holdout_cols + [c for c in optional_diversity_cols if c in holdout_schema_cols]

holdout = pd.read_parquet(HOLDOUT_PATH, columns=read_cols)
baseline = pd.read_parquet(BASELINE_PATH, columns=['user_id', 'video_id', 'score_vanilla'])
debiased = pd.read_parquet(DEBIASED_EM06_PATH, columns=['user_id', 'video_id', 'EU_hat'])

scores = (
    holdout
    .merge(baseline, on=['user_id', 'video_id'], how='left', validate='one_to_one')
    .merge(debiased, on=['user_id', 'video_id'], how='left', validate='one_to_one')
)

missing_baseline = scores['score_vanilla'].isna().sum()
missing_debiased = scores['EU_hat'].isna().sum()
if missing_baseline or missing_debiased:
    raise ValueError(f'missing scores: baseline={missing_baseline:,}, debiased={missing_debiased:,}')

rng = np.random.default_rng(RANDOM_SEED)
scores['random_score'] = rng.random(len(scores)).astype('float32')
scores['oracle_score'] = scores['u_star'].astype('float64')

scores['n_candidates_user'] = scores.groupby('user_id')['video_id'].transform('size').astype('int32')
expected_n = scores['n_candidates_user'].iloc[0]
if not (scores['n_candidates_user'] == expected_n).all():
    print('warning: candidate set size varies by user')

print('rows:', f'{len(scores):,}')
print('users:', scores['user_id'].nunique())
print('videos:', scores['video_id'].nunique())
print('candidate rows per user:')
display(scores.groupby('user_id').size().describe().to_frame('holdout_candidates_per_user').T)
display(scores[['u_star', 'score_vanilla', 'EU_hat', 'random_score']].describe().T)


rows: 1,777,000
users: 1777
videos: 10728
candidate rows per user:


,count,mean,std,min,25%,50%,75%,max
holdout_candidates_per_user,1777.0,1000.0,0.0,1000.0,1000.0,1000.0,1000.0,1000.0


,count,mean,std,min,25%,50%,75%,max
u_star,1777000.0,1.952326,1.940389,-1.788335e+01,0.610872,1.989707,3.353827,23.315222
score_vanilla,1777000.0,-0.040319,0.036573,-3.074607e-01,-0.063986,-0.040922,-0.012282,0.048783
EU_hat,1777000.0,1.534101,1.364178,-2.382624e+00,0.435904,1.655220,2.629906,6.327780
random_score,1777000.0,0.500087,0.288573,2.612286e-07,0.249818,0.500105,0.750047,1.000000


## 2. Ranking Setup

This section defines the method dictionary, cutoff values, shifted NDCG gains, top-decile relevance labels, and ideal DCG denominators.

All metrics are macro-averaged across users:

$$
\frac{1}{|I|}\sum_{i\in I}\mathrm{metric}_i.
$$

Macro-averaging gives every validation user equal weight, regardless of the number of original interactions they had before simulation.


In [3]:
METHODS = {
    'oracle': {
        'score_col': 'oracle_score',
        'description': 'rank by realized true utility u_star',
    },
    'random': {
        'score_col': 'random_score',
        'description': 'rank by seeded uniform random score',
    },
    'baseline_mlp': {
        'score_col': 'score_vanilla',
        'description': 'notebook 10b behavior-head MLP vanilla score',
    },
    'debiased_em06_eu_mlp': {
        'score_col': 'EU_hat',
        'description': 'new notebook 10c EM06 posterior-EU MLP score',
    },
}

MAX_K = max(KS)
USER_COL = 'user_id'
ITEM_COL = 'video_id'
TARGET_COL = 'u_star'

# User-local nonnegative gains for NDCG.
user_min_utility = scores.groupby(USER_COL)[TARGET_COL].transform('min')
scores['utility_gain_shifted'] = (scores[TARGET_COL] - user_min_utility).astype('float64')

# Top-decile relevance labels per user.
utility_rank_desc = scores.groupby(USER_COL)[TARGET_COL].rank(method='first', ascending=False)
relevant_cutoff = np.ceil(scores['n_candidates_user'] * RELEVANCE_TOP_FRAC).astype('int32')
scores['is_top_decile_utility'] = (utility_rank_desc <= relevant_cutoff).astype('int8')
scores['oracle_rank'] = utility_rank_desc.astype('int32')

# Ideal NDCG denominators from true-utility ranking.
oracle_sorted = scores.sort_values([USER_COL, TARGET_COL, ITEM_COL], ascending=[True, False, True]).copy()
oracle_sorted['oracle_rank_order'] = oracle_sorted.groupby(USER_COL).cumcount().add(1).astype('int32')
oracle_top = oracle_sorted.loc[oracle_sorted['oracle_rank_order'] <= MAX_K, [USER_COL, ITEM_COL, 'oracle_rank_order', 'utility_gain_shifted']].copy()
oracle_top['discount'] = 1.0 / np.log2(oracle_top['oracle_rank_order'].to_numpy(dtype='float64') + 1.0)
oracle_top['idcg_contrib'] = oracle_top['utility_gain_shifted'] * oracle_top['discount']

idcg_by_k = {}
for k in KS:
    idcg = oracle_top.loc[oracle_top['oracle_rank_order'] <= k].groupby(USER_COL)['idcg_contrib'].sum()
    idcg_by_k[k] = idcg

num_users = scores[USER_COL].nunique()
relevant_per_user = scores.groupby(USER_COL)['is_top_decile_utility'].sum().rename('n_relevant')

print('users:', num_users)
print('max_k:', MAX_K)
print('top-decile relevant items per user:')
display(relevant_per_user.describe().to_frame().T)


users: 1777
max_k: 100
top-decile relevant items per user:


,count,mean,std,min,25%,50%,75%,max
n_relevant,1777.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0


In [4]:
def rank_top(scores_df: pd.DataFrame, score_col: str) -> pd.DataFrame:
    ranked = scores_df.sort_values([USER_COL, score_col, ITEM_COL], ascending=[True, False, True]).copy()
    ranked['rank'] = ranked.groupby(USER_COL).cumcount().add(1).astype('int32')
    return ranked.loc[ranked['rank'] <= MAX_K, [
        USER_COL, ITEM_COL, TARGET_COL, score_col, 'rank',
        'utility_gain_shifted', 'is_top_decile_utility', 'oracle_rank'
    ]].copy()


def evaluate_method(scores_df: pd.DataFrame, method_name: str, score_col: str):
    top = rank_top(scores_df, score_col)
    top['discount'] = 1.0 / np.log2(top['rank'].to_numpy(dtype='float64') + 1.0)
    top['dcg_contrib'] = top['utility_gain_shifted'] * top['discount']

    summary_rows = []
    per_user_rows = []

    for k in KS:
        sub = top.loc[top['rank'] <= k].copy()
        by_user = sub.groupby(USER_COL).agg(
            mean_utility=(TARGET_COL, 'mean'),
            cumulative_utility=(TARGET_COL, 'sum'),
            dcg=('dcg_contrib', 'sum'),
            relevant_selected=('is_top_decile_utility', 'sum'),
            oracle_overlap_count=('oracle_rank', lambda s: int((s <= k).sum())),
            unique_videos=(ITEM_COL, 'nunique'),
        )
        by_user = by_user.join(idcg_by_k[k].rename('idcg'), how='left')
        by_user = by_user.join(relevant_per_user, how='left')
        by_user['ndcg'] = np.where(by_user['idcg'] > 0, by_user['dcg'] / by_user['idcg'], np.nan)
        by_user['precision_top_decile'] = by_user['relevant_selected'] / float(k)
        by_user['recall_top_decile'] = by_user['relevant_selected'] / by_user['n_relevant'].replace(0, np.nan)
        by_user['hit_rate_top_decile'] = (by_user['relevant_selected'] > 0).astype('float64')
        by_user['oracle_overlap'] = by_user['oracle_overlap_count'] / float(k)

        for user_id, row in by_user.iterrows():
            per_user_rows.append({
                'method': method_name,
                'k': k,
                'user_id': user_id,
                'mean_utility': row['mean_utility'],
                'cumulative_utility': row['cumulative_utility'],
                'ndcg': row['ndcg'],
                'precision_top_decile': row['precision_top_decile'],
                'recall_top_decile': row['recall_top_decile'],
                'hit_rate_top_decile': row['hit_rate_top_decile'],
                'oracle_overlap': row['oracle_overlap'],
            })

        summary_rows.append({
            'method': method_name,
            'score_col': score_col,
            'k': k,
            'mean_utility_at_k': by_user['mean_utility'].mean(),
            'median_user_utility_at_k': by_user['mean_utility'].median(),
            'std_user_utility_at_k': by_user['mean_utility'].std(ddof=1),
            'mean_cumulative_utility_at_k': by_user['cumulative_utility'].mean(),
            'ndcg_at_k': by_user['ndcg'].mean(),
            'precision_top_decile_at_k': by_user['precision_top_decile'].mean(),
            'recall_top_decile_at_k': by_user['recall_top_decile'].mean(),
            'hit_rate_top_decile_at_k': by_user['hit_rate_top_decile'].mean(),
            'oracle_overlap_at_k': by_user['oracle_overlap'].mean(),
            'catalog_coverage_at_k': sub[ITEM_COL].nunique() / scores_df[ITEM_COL].nunique(),
        })

    return pd.DataFrame(summary_rows), pd.DataFrame(per_user_rows), top

summary_parts = []
per_user_parts = []
top_ranked = {}

for method_name, spec in METHODS.items():
    summary, per_user, top = evaluate_method(scores, method_name, spec['score_col'])
    summary_parts.append(summary)
    per_user_parts.append(per_user)
    top_ranked[method_name] = top
    print(f'evaluated {method_name}')

summary = pd.concat(summary_parts, ignore_index=True)
per_user_metrics = pd.concat(per_user_parts, ignore_index=True)

# Add regret and normalized lift after all methods are present.
utility_pivot = summary.pivot(index='k', columns='method', values='mean_utility_at_k')
for idx, row in summary.iterrows():
    k = row['k']
    oracle_u = utility_pivot.loc[k, 'oracle']
    random_u = utility_pivot.loc[k, 'random']
    denom = oracle_u - random_u
    summary.loc[idx, 'regret_at_k'] = oracle_u - row['mean_utility_at_k']
    summary.loc[idx, 'normalized_lift_over_random_at_k'] = (row['mean_utility_at_k'] - random_u) / denom if denom != 0 else np.nan

summary = summary.sort_values(['k', 'method']).reset_index(drop=True)
summary_display = summary[[
    'k', 'method', 'mean_utility_at_k', 'regret_at_k', 'normalized_lift_over_random_at_k',
    'ndcg_at_k', 'precision_top_decile_at_k', 'recall_top_decile_at_k',
    'hit_rate_top_decile_at_k', 'oracle_overlap_at_k', 'catalog_coverage_at_k'
]]

display(summary_display)


evaluated oracle


evaluated random


evaluated baseline_mlp


evaluated debiased_em06_eu_mlp


,k,method,mean_utility_at_k,regret_at_k,normalized_lift_over_random_at_k,ndcg_at_k,precision_top_decile_at_k,recall_top_decile_at_k,hit_rate_top_decile_at_k,oracle_overlap_at_k,catalog_coverage_at_k
0,1,baseline_mlp,2.349103,5.970540,0.071530,0.555235,0.133371,0.001334,0.133371,0.000000,0.039523
1,1,debiased_em06_eu_mlp,2.725334,5.594309,0.130037,0.587230,0.234665,0.002347,0.234665,0.000000,0.098527
2,1,oracle,8.319643,0.000000,1.000000,1.000000,1.000000,0.010000,1.000000,1.000000,0.050336
3,1,random,1.889131,6.430511,0.000000,0.518286,0.093979,0.000940,0.093979,0.000000,0.152871
4,5,baseline_mlp,2.346493,4.036410,0.094096,0.611365,0.127293,0.006365,0.445132,0.003264,0.100764
5,5,debiased_em06_eu_mlp,2.726100,3.656803,0.179293,0.646756,0.227800,0.011390,0.615644,0.009342,0.243755
6,5,oracle,6.382903,0.000000,1.000000,1.000000,1.000000,0.050000,1.000000,1.000000,0.158557
7,5,random,1.927231,4.455672,0.000000,0.573457,0.099043,0.004952,0.410242,0.004277,0.562360
8,10,baseline_mlp,2.340002,3.362823,0.106535,0.641243,0.125098,0.012510,0.648284,0.007147,0.150261
9,10,debiased_em06_eu_mlp,2.704860,2.997965,0.203473,0.677453,0.222172,0.022217,0.805290,0.018796,0.336130


## 3. Score Correlations

Candidate-level correlations ask whether each score is aligned with true utility before taking a top-k cut.

The notebook reports:

| Correlation | Meaning |
|---|---|
| Pearson | linear association between score and `u_star` |
| Spearman | rank association between score and `u_star` |
| pooled candidate correlation | computed over all holdout candidate rows |
| mean/median user correlation | computed within user first, then summarized across users |

A recommender can have strong correlation and still lose top-k utility if ranking errors concentrate near the top of each candidate set, so these correlations complement the top-k metrics rather than replacing them.


In [5]:
corr_rows = []
for method_name, spec in METHODS.items():
    score_col = spec['score_col']
    pearson_all = scores[[score_col, TARGET_COL]].corr(method='pearson').iloc[0, 1]
    spearman_all = scores[[score_col, TARGET_COL]].corr(method='spearman').iloc[0, 1]

    user_corrs = []
    for user_id, g in scores.groupby(USER_COL, sort=False):
        user_corrs.append({
            USER_COL: user_id,
            'pearson': g[[score_col, TARGET_COL]].corr(method='pearson').iloc[0, 1],
            'spearman': g[[score_col, TARGET_COL]].corr(method='spearman').iloc[0, 1],
        })
    user_corrs = pd.DataFrame(user_corrs)

    corr_rows.append({
        'method': method_name,
        'score_col': score_col,
        'description': METHODS[method_name]['description'],
        'pearson_all_candidates': pearson_all,
        'spearman_all_candidates': spearman_all,
        'mean_user_pearson': user_corrs['pearson'].mean(),
        'median_user_pearson': user_corrs['pearson'].median(),
        'mean_user_spearman': user_corrs['spearman'].mean(),
        'median_user_spearman': user_corrs['spearman'].median(),
    })

score_correlations = pd.DataFrame(corr_rows)
display(score_correlations)


,method,score_col,description,pearson_all_candidates,spearman_all_candidates,mean_user_pearson,median_user_pearson,mean_user_spearman,median_user_spearman
0,oracle,oracle_score,rank by realized true utility u_star,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,random,random_score,rank by seeded uniform random score,0.000202,0.000279,0.000765,0.000616,0.000672,-0.000535
2,baseline_mlp,score_vanilla,notebook 10b behavior-head MLP vanilla score,0.672155,0.708241,0.196075,0.213039,0.201904,0.239042
3,debiased_em06_eu_mlp,EU_hat,new notebook 10c EM06 posterior-EU MLP score,0.751301,0.776155,0.415878,0.419696,0.405205,0.429070


## 4. Top-k Comparison Tables

This section pivots the summary metrics so the four policies can be compared side by side at each cutoff. The main welfare comparison is `mean_utility_at_k`; NDCG and top-decile metrics show ranking quality from complementary angles.

The preferred debiased validation comparison is:

$$
\text{debiased EM06 EU MLP} \quad \text{versus} \quad \text{baseline MLP}.
$$

Oracle and random provide the upper and lower reference points for interpreting the scale of the difference.


In [6]:
metric_cols = [
    'mean_utility_at_k',
    'regret_at_k',
    'normalized_lift_over_random_at_k',
    'ndcg_at_k',
    'precision_top_decile_at_k',
    'recall_top_decile_at_k',
    'hit_rate_top_decile_at_k',
    'oracle_overlap_at_k',
]

for metric in metric_cols:
    print('\n' + metric)
    wide = summary.pivot(index='k', columns='method', values=metric)
    ordered_cols = ['oracle', 'debiased_em06_eu_mlp', 'baseline_mlp', 'random']
    display(wide[[c for c in ordered_cols if c in wide.columns]].round(6))



mean_utility_at_k


method,oracle,debiased_em06_eu_mlp,baseline_mlp,random
k,,,,
1,8.319643,2.725334,2.349103,1.889131
5,6.382903,2.726100,2.346493,1.927231
10,5.702825,2.704860,2.340002,1.939027
20,5.122279,2.681857,2.322015,1.948367
50,4.422736,2.631020,2.279059,1.951438
100,3.916251,2.589772,2.249038,1.950280



regret_at_k


method,oracle,debiased_em06_eu_mlp,baseline_mlp,random
k,,,,
1,0.0,5.594309,5.970540,6.430511
5,0.0,3.656803,4.036410,4.455672
10,0.0,2.997965,3.362823,3.763798
20,0.0,2.440422,2.800264,3.173912
50,0.0,1.791717,2.143678,2.471298
100,0.0,1.326480,1.667213,1.965972



normalized_lift_over_random_at_k


...

method,oracle,debiased_em06_eu_mlp,baseline_mlp,random
k,,,,
1,0.01,0.002347,0.001334,0.000940
5,0.05,0.011390,0.006365,0.004952
10,0.10,0.022217,0.012510,0.010000
20,0.20,0.043033,0.024344,0.020428
50,0.50,0.099297,0.058301,0.050720
100,1.00,0.191474,0.114733,0.100355



hit_rate_top_decile_at_k


method,oracle,debiased_em06_eu_mlp,baseline_mlp,random
k,,,,
1,1.0,0.234665,0.133371,0.093979
5,1.0,0.615644,0.445132,0.410242
10,1.0,0.805290,0.648284,0.662352
20,1.0,0.933033,0.839055,0.889139
50,1.0,0.997749,0.981992,0.997186
100,1.0,1.000000,0.999437,1.000000



oracle_overlap_at_k


method,oracle,debiased_em06_eu_mlp,baseline_mlp,random
k,,,,
1,1.0,0.000000,0.000000,0.000000
5,1.0,0.009342,0.003264,0.004277
10,1.0,0.018796,0.007147,0.009173
20,1.0,0.035903,0.012859,0.020034
50,1.0,0.087586,0.044547,0.050264
100,1.0,0.191474,0.114733,0.100355


## 5. Incremental Gain over Baseline

This section subtracts baseline metrics from debiased EM06 metrics at each cutoff:

$$
\Delta_k = \mathrm{Metric@k}(debiased)-\mathrm{Metric@k}(baseline).
$$

For utility, NDCG, lift, precision, recall, hit rate, and oracle overlap, positive values favor the debiased recommender. For regret, negative values favor the debiased recommender because lower regret is better.


In [7]:
comparison_rows = []
wide_summary = summary.pivot(index='k', columns='method')

for k in KS:
    row = {'k': k}
    for metric in metric_cols:
        deb = wide_summary.loc[k, (metric, 'debiased_em06_eu_mlp')]
        base = wide_summary.loc[k, (metric, 'baseline_mlp')]
        row[f'{metric}_debiased_minus_baseline'] = deb - base
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(6))


,k,mean_utility_at_k_debiased_minus_baseline,regret_at_k_debiased_minus_baseline,normalized_lift_over_random_at_k_debiased_minus_baseline,ndcg_at_k_debiased_minus_baseline,precision_top_decile_at_k_debiased_minus_baseline,recall_top_decile_at_k_debiased_minus_baseline,hit_rate_top_decile_at_k_debiased_minus_baseline,oracle_overlap_at_k_debiased_minus_baseline
0,1,0.376231,-0.376231,0.058507,0.031995,0.101294,0.001013,0.101294,0.000000
1,5,0.379606,-0.379606,0.085196,0.035392,0.100506,0.005025,0.170512,0.006078
2,10,0.364858,-0.364858,0.096939,0.036210,0.097074,0.009707,0.157006,0.011649
3,20,0.359841,-0.359841,0.113375,0.037657,0.093444,0.018689,0.093979,0.023044
4,50,0.351961,-0.351961,0.142419,0.039723,0.081992,0.040996,0.015757,0.043039
5,100,0.340733,-0.340733,0.173315,0.041027,0.076742,0.076742,0.000563,0.076742


## 6. Save Evaluation Outputs

This section saves the evaluation bundle:

| Output | Contents |
|---|---|
| `validation_recsys_ranking_metrics_summary.csv` | method-by-cutoff summary metrics |
| `validation_recsys_ranking_metrics_per_user.parquet` | user-level metrics for heterogeneity analysis |
| `validation_recsys_score_correlations.csv` | pooled and user-level Pearson/Spearman correlations |
| `validation_recsys_evaluation_metrics.json` | machine-readable evaluation report with inputs and outputs |

These files are the reporting layer for the validation experiment.


In [8]:
summary.to_csv(SUMMARY_CSV, index=False)
per_user_metrics.to_parquet(PER_USER_PARQUET, index=False)
score_correlations.to_csv(CORR_CSV, index=False)

payload = {
    'random_seed': RANDOM_SEED,
    'ks': KS,
    'relevance_top_frac': RELEVANCE_TOP_FRAC,
    'num_rows': int(len(scores)),
    'num_users': int(scores[USER_COL].nunique()),
    'num_videos': int(scores[ITEM_COL].nunique()),
    'input_paths': {
        'holdout_candidates': str(HOLDOUT_PATH),
        'baseline_holdout_scores': str(BASELINE_PATH),
        'debiased_em06_holdout_scores': str(DEBIASED_EM06_PATH),
        'baseline_metrics': str(BASELINE_METRICS_PATH),
        'debiased_em06_metrics': str(DEBIASED_EM06_METRICS_PATH),
    },
    'output_paths': {
        'summary_csv': str(SUMMARY_CSV),
        'per_user_parquet': str(PER_USER_PARQUET),
        'score_correlations_csv': str(CORR_CSV),
    },
    'method_descriptions': METHODS,
    'summary_records': summary.to_dict(orient='records'),
    'score_correlation_records': score_correlations.to_dict(orient='records'),
}
EVAL_JSON.write_text(json.dumps(payload, indent=2, allow_nan=True))

print('saved summary:', SUMMARY_CSV)
print('saved per-user metrics:', PER_USER_PARQUET)
print('saved score correlations:', CORR_CSV)
print('saved json:', EVAL_JSON)


saved summary: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_recsys_ranking_metrics_summary.csv
saved per-user metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_recsys_ranking_metrics_per_user.parquet
saved score correlations: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_recsys_score_correlations.csv
saved json: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_recsys_evaluation_metrics.json


## 7. Reading the Results

Use this checklist when interpreting the tables:

1. `mean_utility_at_k`: higher means the policy delivers more realized true utility.
2. `regret_at_k`: lower means closer to oracle.
3. `normalized_lift_over_random_at_k`: higher means the policy recovers more of the oracle improvement over random.
4. `ndcg_at_k`: higher means high-utility videos appear earlier in the ranked list.
5. `precision_top_decile_at_k` and `recall_top_decile_at_k`: higher means the policy retrieves more videos from each user's high-utility tail.
6. `oracle_overlap_at_k`: higher means the policy selects the same videos as the oracle top-k set.
7. `catalog_coverage_at_k`: higher means recommendations use a broader set of videos across users; this is a diversity diagnostic, not a welfare metric by itself.
